In [ ]:
# ==========================================
# CELL 1: SETUP, LABELS, DATASET, SPLIT
# ==========================================
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# --- MODIFIED: Using relative paths for GitHub Repo ---
# Assuming this notebook is inside a "notebooks/" folder
IMAGE_FOLDER = r"../data/Ready_for_traning"
CSV_FILE     = r"../data/metadata_embed.csv"
MASTER_CSV   = r"../data/embed_merged_without_blank_tissueden.csv"

STAGE1_MODEL_PATH = r"../outputs/convnext_stage1_best.pth"
STAGE2_MODEL_PATH = r"../outputs/convnext_stage2_best.pth"
OUTPUT_FEATURES   = r"../outputs/convnext_finetuned_features.npy"
OUTPUT_FILENAMES  = r"../outputs/convnext_finetuned_filenames.npy"
OUTPUT_LABELS     = r"../outputs/convnext_finetuned_labels.npy"
# ------------------------------------------------------

IMAGE_SIZE          = 1024
BATCH_SIZE          = 16       # increased: T4 x2 can handle it
NUM_WORKERS         = 2
NUM_CLASSES         = 4
EPOCHS_STAGE1       = 10       # increased: was 3
EPOCHS_STAGE2       = 15       # increased: was 3
LR_HEAD_STAGE1      = 1e-3
LR_HEAD_STAGE2      = 3e-4     # increased: was 1e-4
LR_BACKBONE_STAGE2  = 3e-5     # increased: was 1e-5
WEIGHT_DECAY        = 1e-4
VAL_SIZE            = 0.2
RANDOM_STATE        = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device set to: {DEVICE}")

LAT_MAP = {'L': 'L', 'LEFT': 'L', 'R': 'R', 'RIGHT': 'R'}

# ── Load metadata CSV (laterality only) ──────────────────────
meta_df = pd.read_csv(CSV_FILE, low_memory=False)
meta_df["base_id"]   = meta_df["filename"].astype(str).str.replace(".dcm", "", regex=False)
meta_df["lat_clean"] = meta_df["laterality"].astype(str).str.strip().str.upper().map(LAT_MAP).fillna("UNKNOWN")

# ── Load master CSV (labels only) ────────────────────────────
master_df = pd.read_csv(MASTER_CSV, low_memory=False)
master_df["base_id"] = master_df["anon_dicom_path"].astype(str).apply(
    lambda p: os.path.splitext(os.path.basename(p))[0]
)
master_df["label"] = master_df["tissueden"].astype(int) - 1

# ── Collect PNG paths ─────────────────────────────────────────
png_paths = sorted(
    os.path.join(root, f)
    for root, _, files in os.walk(IMAGE_FOLDER)
    for f in files if f.lower().endswith(".png")
)

png_df = pd.DataFrame({"img_path": png_paths})
png_df["base_id"] = png_df["img_path"].apply(lambda p: os.path.splitext(os.path.basename(p))[0])

# ── Merge: PNG → laterality → label ──────────────────────────
records = png_df.merge(
    meta_df[["base_id", "lat_clean"]],
    on="base_id", how="inner"
).merge(
    master_df[["base_id", "label"]],
    on="base_id", how="inner"
).drop_duplicates(subset=["img_path"]).reset_index(drop=True)

print(f"Matched records: {len(records)}")
print(records["label"].value_counts().sort_index())

train_df, val_df = train_test_split(
    records,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=records["label"]
)

class SmartMammogramDataset(Dataset):
    def __init__(self, records_df, transform=None):
        self.records = records_df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        img_path   = row["img_path"]
        laterality = row["lat_clean"]
        label      = int(row["label"])

        img = Image.open(img_path).convert("RGB")
        if laterality == "R":
            img = img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)

        if self.transform:
            img = self.transform(img)

        return img, label, img_path

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_dataset = SmartMammogramDataset(train_df, transform=train_transform)
val_dataset   = SmartMammogramDataset(val_df,   transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

print(f"Train images: {len(train_dataset)}")
print(f"Val images  : {len(val_dataset)}")

In [ ]:
# ==========================================
# CELL 2: STAGE 1 TRAINING
# ==========================================
model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(DEVICE)

for param in model.features.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    model = nn.DataParallel(model)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD_STAGE1,
    weight_decay=WEIGHT_DECAY
)

best_val_acc = 0.0

for epoch in range(EPOCHS_STAGE1):
    model.train()
    train_correct = 0
    train_total = 0
    train_loss = 0.0

    for imgs, labels, _ in tqdm(train_loader, desc=f"Stage 1 Epoch {epoch+1}"):
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0.0

    with torch.inference_mode():
        for imgs, labels, _ in val_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits = model(imgs)
            loss = criterion(logits, labels)

            val_loss += loss.item() * imgs.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}: train_acc={train_acc:.4f}, val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), STAGE1_MODEL_PATH)

print(f"Best Stage 1 val acc: {best_val_acc:.4f}")

In [ ]:
# ==========================================
# CELL 3: STAGE 2 FINE-TUNING
# ==========================================
model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)
model.load_state_dict(torch.load(STAGE1_MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)

for param in model.features.parameters():
    param.requires_grad = False
for param in model.features[-2:].parameters():   # unfreeze last 2 blocks (was -1)
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

backbone_params = []
head_params = []

for name, param in model.named_parameters():
    if param.requires_grad:
        if "classifier" in name:
            head_params.append(param)
        else:
            backbone_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": LR_BACKBONE_STAGE2},
        {"params": head_params,     "lr": LR_HEAD_STAGE2},
    ],
    weight_decay=WEIGHT_DECAY
)

criterion = nn.CrossEntropyLoss()
best_val_acc = 0.0

for epoch in range(EPOCHS_STAGE2):
    model.train()
    train_correct = 0
    train_total   = 0
    train_loss    = 0.0

    for imgs, labels, _ in tqdm(train_loader, desc=f"Stage 2 Epoch {epoch+1}"):
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item() * imgs.size(0)
        preds          = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total   += labels.size(0)

    train_loss /= train_total
    train_acc   = train_correct / train_total

    model.eval()
    val_correct = 0
    val_total   = 0
    val_loss    = 0.0

    with torch.inference_mode():
        for imgs, labels, _ in val_loader:
            imgs   = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits = model(imgs)
            loss   = criterion(logits, labels)

            val_loss    += loss.item() * imgs.size(0)
            preds        = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)

    val_loss /= val_total
    val_acc   = val_correct / val_total

    print(f"Epoch {epoch+1}: train_acc={train_acc:.4f}, val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        
        # Ensure output directory exists before saving
        os.makedirs(os.path.dirname(STAGE2_MODEL_PATH), exist_ok=True)
        torch.save(model.state_dict(), STAGE2_MODEL_PATH)

print(f"Best Stage 2 val acc: {best_val_acc:.4f}")

In [ ]:
# ==========================================
# CELL 4: EXPORT FINETUNED FEATURES
# ==========================================
feature_model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
in_features = feature_model.classifier[2].in_features
feature_model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)
feature_model.load_state_dict(torch.load(STAGE2_MODEL_PATH, map_location=DEVICE))
feature_model.classifier = nn.Flatten(start_dim=1)
feature_model = feature_model.to(DEVICE)
feature_model.eval()

full_dataset = SmartMammogramDataset(records, transform=val_transform)
full_loader = DataLoader(
    full_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

all_features = []
all_filenames = []
all_labels = []

with torch.inference_mode(), torch.autocast(device_type=DEVICE.type):
    for imgs, labels, paths in tqdm(full_loader, desc="Exporting Features", unit="batch"):
        imgs = imgs.to(DEVICE, non_blocking=True)
        feats = feature_model(imgs)
        all_features.append(feats.cpu().float().numpy())
        all_labels.append(labels.numpy())
        all_filenames.extend(paths)

all_features = np.vstack(all_features)
all_labels = np.concatenate(all_labels)

# Ensure output directory exists before saving features
os.makedirs(os.path.dirname(OUTPUT_FEATURES), exist_ok=True)

np.save(OUTPUT_FEATURES, all_features)
np.save(OUTPUT_FILENAMES, np.array(all_filenames))
np.save(OUTPUT_LABELS, all_labels)

print(f"Feature matrix shape: {all_features.shape}")
print(f"Saved features to   : {OUTPUT_FEATURES}")
print(f"Saved filenames to  : {OUTPUT_FILENAMES}")
print(f"Saved labels to     : {OUTPUT_LABELS}")